# Dynamic Topology Mixture of Experts: End-to-End Implementation

This notebook implements a revolutionary Dynamic Topology MoE layer where each expert can adaptively change its neural network architecture during training.

## Key Innovations:
- **Structurally Adaptive Experts**: Each expert evolves its depth and width based on input patterns
- **Dynamic Memory Management**: Pre-allocated memory pools with intelligent masking
- **Capacity-Aware Routing**: Routing considers expert capacities and current loads
- **Stable Training**: Hard limits and regularization prevent architectural collapse

## Architecture Overview:
- **Expert₁**: Shallow, wide network (2 layers, 4096 neurons) for simple patterns
- **Expert₂**: Deep, narrow network (6 layers, 512 neurons) for complex reasoning
- **Expert₃**: Pyramid structure (4 layers: 2048→1024→512→256) for hierarchical processing



In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import numpy as np
from dataclasses import dataclass
from tqdm.auto import tqdm
from contextlib import nullcontext
import os
import matplotlib.pyplot as plt
from typing import Dict, List, Tuple, Optional
import warnings
warnings.filterwarnings('ignore')

# Set device
device = "cuda" if torch.cuda.is_available() else "cpu"
device_type = 'cuda' if 'cuda' in device else 'cpu'
dtype = 'bfloat16' if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else 'float16'
ptdtype = {'float32': torch.float32, 'bfloat16': torch.bfloat16, 'float16': torch.float16}[dtype]
ctx = nullcontext() if device_type == 'cpu' else torch.amp.autocast(device_type=device_type, dtype=ptdtype)

torch.set_default_device(device)
torch.manual_seed(42)

print(f"Using device: {device}")
print(f"Data type: {dtype}")

## Dynamic Memory Pool Implementation

This cell implements the core memory management system for dynamic expert architectures.

### Key Features:
- **Pre-allocated Memory**: Allocates maximum possible memory upfront to avoid fragmentation
- **Boolean Masks**: Uses simple boolean masks to track active layers and neurons
- **Efficient Access**: Provides fast access to expert weights through masking operations

### Architecture Management:
- `get_expert_weights()`: Retrieves only active weights for a specific expert layer
- `set_expert_architecture()`: Updates expert architecture by modifying masks
- `get_expert_architecture()`: Returns current depth and width configuration

### Memory Efficiency:
- Zero memory allocation during training
- Contiguous memory layout for GPU efficiency
- Simple tensor slicing operations

In [ ]:
class DynamicMemoryPool:
    """
    Pre-allocated memory pool for dynamic expert architectures.
    Uses masking to handle variable-sized experts efficiently.
    """
    def __init__(self, num_experts: int, max_layers: int, max_neurons: int, hidden_size: int):
        self.num_experts = num_experts
        self.max_layers = max_layers
        self.max_neurons = max_neurons
        self.hidden_size = hidden_size
        
        # Pre-allocate maximum possible memory
        self.weight_pool = nn.Parameter(
            torch.randn(num_experts, max_layers, max_neurons, max_neurons) * 0.02
        )
        self.bias_pool = nn.Parameter(
            torch.zeros(num_experts, max_layers, max_neurons)
        )
        
        # Track active components with masks
        self.active_layers = torch.zeros(num_experts, max_layers, dtype=torch.bool, device=device)
        self.active_neurons = torch.zeros(num_experts, max_layers, max_neurons, dtype=torch.bool, device=device)
        
        # Initialize first layer for each expert
        for expert_id in range(num_experts):
            self.active_layers[expert_id, 0] = True
            self.active_neurons[expert_id, 0, :512] = True  # Start with 512 neurons
    
    def get_expert_weights(self, expert_id: int, layer_id: int) -> Tuple[torch.Tensor, torch.Tensor]:
        """Get active weights and bias for specific expert layer"""
        if not self.active_layers[expert_id, layer_id]:
            return None, None
            
        layer_mask = self.active_neurons[expert_id, layer_id]
        active_neurons = layer_mask.sum().item()
        
        if active_neurons == 0:
            return None, None
            
        # Get active weights and bias
        active_weights = self.weight_pool[expert_id, layer_id][:active_neurons, :active_neurons]
        active_bias = self.bias_pool[expert_id, layer_id][:active_neurons]
        
        return active_weights, active_bias
    
    def set_expert_architecture(self, expert_id: int, depths: List[int], widths: List[int]):
        """Set the architecture for a specific expert"""
        # Reset all layers for this expert
        self.active_layers[expert_id, :] = False
        self.active_neurons[expert_id, :, :] = False
        
        # Set active layers and neurons
        for layer_id, width in enumerate(widths):
            if layer_id < self.max_layers:
                self.active_layers[expert_id, layer_id] = True
                self.active_neurons[expert_id, layer_id, :width] = True
    
    def get_expert_architecture(self, expert_id: int) -> Tuple[int, List[int]]:
        """Get current architecture for an expert"""
        active_layers = self.active_layers[expert_id].nonzero().squeeze(-1)
        depth = len(active_layers)
        
        widths = []
        for layer_id in active_layers:
            width = self.active_neurons[expert_id, layer_id].sum().item()
            widths.append(width)
            
        return depth, widths
    
    def get_all_architectures(self) -> Dict[int, Tuple[int, List[int]]]:
        """Get architectures for all experts"""
        architectures = {}
        for expert_id in range(self.num_experts):
            architectures[expert_id] = self.get_expert_architecture(expert_id)
        return architectures

## Dynamic Gradient Flow Management

This cell handles the critical challenge of backpropagation through variable-depth networks.

### Key Innovations:
- **Path-Length Normalization**: Scales gradients by √depth to maintain stable magnitudes
- **Smooth Transitions**: Handles architectural changes without disrupting training
- **Information Preservation**: Merges information when layers are removed

### Growth Strategy:
- Initializes new layers with scaled versions of existing weights
- Uses small initialization scale (0.1) to prevent disruption
- Maintains gradient flow continuity

### Shrinkage Strategy:
- Averages weights from removed layers into remaining ones
- Uses blend factor (0.3) to preserve learned information
- Prevents catastrophic forgetting during architecture changes

In [ ]:
class DynamicGradientFlow:
    """
    Handles gradient flow through variable-depth expert networks.
    Normalizes gradients by path length and manages architectural transitions.
    """
    def __init__(self):
        self.gradient_scaling = {}
        self.architecture_history = {}
        
    def normalize_gradients(self, expert_id: int, gradients: torch.Tensor, current_depth: int) -> torch.Tensor:
        """Normalize gradients by effective path length"""
        scale_factor = 1.0 / math.sqrt(current_depth)
        normalized_grads = gradients * scale_factor
        self.gradient_scaling[expert_id] = scale_factor
        return normalized_grads
        
    def handle_architecture_change(self, expert_id: int, old_depth: int, new_depth: int, 
                                 memory_pool: DynamicMemoryPool):
        """Handle smooth transitions when expert architecture changes"""
        if new_depth > old_depth:  # Growth
            self._initialize_new_layers(expert_id, old_depth, new_depth, memory_pool)
        elif new_depth < old_depth:  # Shrinkage
            self._merge_layer_information(expert_id, old_depth, new_depth, memory_pool)
            
        # Update history
        self.architecture_history[expert_id] = {
            'old_depth': old_depth,
            'new_depth': new_depth,
            'step': len(self.architecture_history.get(expert_id, {}).get('changes', []))
        }
            
    def _initialize_new_layers(self, expert_id: int, old_depth: int, new_depth: int, 
                              memory_pool: DynamicMemoryPool):
        """Initialize new layers with scaled versions of existing ones"""
        if old_depth == 0:
            return
            
        # Get last layer weights as template
        last_weights, last_bias = memory_pool.get_expert_weights(expert_id, old_depth - 1)
        if last_weights is None:
            return
            
        init_scale = 0.1  # Small initialization for new layers
        
        for new_layer_id in range(old_depth, new_depth):
            # Initialize new layer with scaled version of last layer
            new_size = last_weights.shape[0]  # Use same size initially
            new_weights = torch.randn(new_size, new_size, device=device) * 0.02 * init_scale
            new_bias = torch.zeros(new_size, device=device)
            
            # Copy to memory pool
            memory_pool.weight_pool.data[expert_id, new_layer_id, :new_size, :new_size] = new_weights
            memory_pool.bias_pool.data[expert_id, new_layer_id, :new_size] = new_bias
            
    def _merge_layer_information(self, expert_id: int, old_depth: int, new_depth: int, 
                                memory_pool: DynamicMemoryPool):
        """Merge information from removed layers into remaining ones"""
        # Simple approach: average the weights of removed layers into the last remaining layer
        if new_depth == 0 or old_depth <= new_depth:
            return
            
        # Get the last remaining layer
        last_weights, last_bias = memory_pool.get_expert_weights(expert_id, new_depth - 1)
        if last_weights is None:
            return
            
        # Average weights from removed layers
        removed_weights = []
        for layer_id in range(new_depth, old_depth):
            weights, bias = memory_pool.get_expert_weights(expert_id, layer_id)
            if weights is not None:
                removed_weights.append(weights)
                
        if removed_weights:
            avg_removed = torch.stack(removed_weights).mean(0)
            # Blend with existing weights
            blend_factor = 0.3
            new_weights = (1 - blend_factor) * last_weights + blend_factor * avg_removed
            memory_pool.weight_pool.data[expert_id, new_depth - 1, :last_weights.shape[0], :last_weights.shape[1]] = new_weights

## Simple Architectural Controller

This cell implements the decision-making system for when and how to modify expert architectures.

### Decision Logic:
- **Growth Threshold (0.8)**: Expert grows when top layer utilization > 80%
- **Shrink Threshold (0.2)**: Expert shrinks when top layer utilization < 20%
- **Cooldown Period (1000 steps)**: Prevents rapid oscillations

### Stability Mechanisms:
- **Exponential Moving Average (EMA)**: Smooths utilization metrics with α=0.9
- **Hard Limits**: Enforces min/max layers and neurons
- **Change Cooldown**: Prevents too frequent architectural changes

### Utilization Tracking:
- Monitors activation magnitudes for each layer
- Updates EMA for stable decision-making
- Tracks architectural change history

In [ ]:
class SimpleArchitecturalController:
    """
    Simple threshold-based controller for architectural decisions.
    Uses exponential moving averages for stability.
    """
    def __init__(self, growth_threshold: float = 0.8, shrink_threshold: float = 0.2,
                 min_layers: int = 1, max_layers: int = 6, min_neurons: int = 64, max_neurons: int = 2048):
        self.growth_threshold = growth_threshold
        self.shrink_threshold = shrink_threshold
        self.min_layers = min_layers
        self.max_layers = max_layers
        self.min_neurons = min_neurons
        self.max_neurons = max_neurons
        
        # Exponential moving averages for utilization
        self.utilization_ema = {}
        self.alpha = 0.9  # EMA smoothing factor
        
        # Change cooldown to prevent rapid oscillations
        self.change_cooldown = {}
        self.cooldown_steps = 1000
        
    def update_utilization(self, expert_id: int, layer_id: int, activation_mean: float):
        """Update utilization with exponential moving average"""
        key = (expert_id, layer_id)
        
        if key not in self.utilization_ema:
            self.utilization_ema[key] = activation_mean
        else:
            self.utilization_ema[key] = (self.alpha * self.utilization_ema[key] + 
                                        (1 - self.alpha) * activation_mean)
    
    def should_grow_expert(self, expert_id: int, current_depth: int, current_step: int) -> bool:
        """Decide if expert should grow"""
        if current_depth >= self.max_layers:
            return False
            
        # Check cooldown
        if not self._can_change_architecture(expert_id, current_step):
            return False
            
        # Check top layer utilization
        top_layer = current_depth - 1
        utilization = self.utilization_ema.get((expert_id, top_layer), 0.5)
        
        return utilization > self.growth_threshold
    
    def should_shrink_expert(self, expert_id: int, current_depth: int, current_step: int) -> bool:
        """Decide if expert should shrink"""
        if current_depth <= self.min_layers:
            return False
            
        # Check cooldown
        if not self._can_change_architecture(expert_id, current_step):
            return False
            
        # Check top layer utilization
        top_layer = current_depth - 1
        utilization = self.utilization_ema.get((expert_id, top_layer), 0.5)
        
        return utilization < self.shrink_threshold
        
    def should_adjust_width(self, expert_id: int, layer_id: int, current_width: int) -> str:
        """Decide if layer width should be adjusted"""
        utilization = self.utilization_ema.get((expert_id, layer_id), 0.5)
        
        if utilization > 0.9 and current_width < self.max_neurons:
            return "grow"
        elif utilization < 0.1 and current_width > self.min_neurons:
            return "shrink"
        else:
            return "maintain"
            
    def _can_change_architecture(self, expert_id: int, current_step: int) -> bool:
        """Check if enough time has passed since last change"""
        last_change = self.change_cooldown.get(expert_id, -self.cooldown_steps)
        return current_step - last_change > self.cooldown_steps
        
    def record_architecture_change(self, expert_id: int, current_step: int):
        """Record when architecture was changed"""
        self.change_cooldown[expert_id] = current_step

## Capacity-Aware Load Balancer

This cell implements intelligent load balancing that accounts for different expert capacities.

### Key Innovation:
- **Capacity Normalization**: Adjusts expected loads based on expert parameter counts
- **Dynamic Routing**: Modifies routing probabilities to favor higher-capacity experts
- **Cache Management**: Efficiently tracks expert capacities with invalidation

### Load Balance Loss:
- Computes actual vs expected token distribution
- Normalizes by expert capacity ratios
- Ensures fair computational distribution

### Routing Adjustment:
- Higher capacity experts receive more tokens
- Maintains overall routing probability distribution
- Adapts to architectural changes automatically

In [ ]:
class CapacityAwareLoadBalancer:
    """
    Load balancer that accounts for different expert capacities.
    Normalizes load by expert capacity to ensure fair distribution.
    """
    def __init__(self, num_experts: int):
        self.num_experts = num_experts
        self.capacity_cache = {}
        self.cache_valid = False
        
    def compute_expert_capacity(self, expert_id: int, architectures: Dict) -> int:
        """Compute capacity as total active neurons across layers"""
        if not self.cache_valid:
            self.capacity_cache.clear()
            
        if expert_id in self.capacity_cache:
            return self.capacity_cache[expert_id]
            
        if expert_id not in architectures:
            return 1  # Default capacity
            
        depth, widths = architectures[expert_id]
        total_neurons = sum(widths)
        self.capacity_cache[expert_id] = total_neurons
        return total_neurons
    
    def compute_load_balance_loss(self, routing_probs: torch.Tensor, 
                                 expert_assignments: torch.Tensor,
                                 architectures: Dict) -> torch.Tensor:
        """Compute capacity-aware load balancing loss"""
        # Compute actual loads
        actual_loads = torch.bincount(expert_assignments, minlength=self.num_experts)
        
        # Compute capacities for all experts
        capacities = torch.tensor([
            self.compute_expert_capacity(i, architectures) for i in range(self.num_experts)
        ], device=device)
        
        # Normalize capacities to sum to num_experts (maintains scale)
        normalized_capacities = capacities * self.num_experts / (capacities.sum() + 1e-8)
        
        # Expected load should be proportional to capacity
        expected_loads = routing_probs.sum(0) * normalized_capacities
        
        # Standard load balance loss with capacity adjustment
        load_balance_loss = ((actual_loads - expected_loads) ** 2).mean()
        
        return load_balance_loss
        
    def get_routing_adjustment(self, base_routing_probs: torch.Tensor, 
                              architectures: Dict) -> torch.Tensor:
        """Adjust routing probabilities for capacity"""
        capacities = torch.tensor([
            self.compute_expert_capacity(i, architectures) for i in range(self.num_experts)
        ], device=device)
        
        # Higher capacity experts can handle more load
        capacity_weights = capacities / (capacities.mean() + 1e-8)
        adjusted_probs = base_routing_probs * capacity_weights
        
        return torch.softmax(adjusted_probs, dim=-1)
        
    def invalidate_cache(self):
        """Invalidate capacity cache when architectures change"""
        self.cache_valid = False

## Stability Controller

This cell prevents architectural collapse and ensures training stability.

### Hard Limits:
- **Depth Limits**: 1-6 layers per expert
- **Width Limits**: 64-2048 neurons per layer
- **Change Rate Limits**: Prevents unbounded growth/shrinkage

### Regularization:
- **Diversity Loss**: Encourages different expert architectures
- **Size Regularization**: Targets optimal parameter count (~1M per expert)
- **Similarity Penalty**: Prevents experts from converging to identical structures

### Architecture Similarity:
- Compares depth and width distributions
- Normalizes similarity scores
- Encourages architectural diversity

In [ ]:
class StabilityController:
    """
    Prevents architectural collapse and ensures training stability.
    Enforces hard limits and provides gentle regularization.
    """
    def __init__(self, min_layers: int = 1, max_layers: int = 6, 
                 min_neurons: int = 64, max_neurons: int = 2048):
        self.min_layers = min_layers
        self.max_layers = max_layers
        self.min_neurons = min_neurons
        self.max_neurons = max_neurons
        
    def enforce_architectural_limits(self, expert_id: int, proposed_depth: int, 
                                   proposed_widths: List[int]) -> Tuple[int, List[int]]:
        """Enforce hard limits to prevent collapse/explosion"""
        # Depth limits
        actual_depth = max(self.min_layers, min(self.max_layers, proposed_depth))
        
        # Width limits per layer
        actual_widths = []
        for width in proposed_widths[:actual_depth]:
            actual_width = max(self.min_neurons, min(self.max_neurons, width))
            actual_widths.append(actual_width)
            
        return actual_depth, actual_widths
    
    def compute_stability_loss(self, expert_architectures: Dict) -> torch.Tensor:
        """Compute regularization loss for architectural stability"""
        if not expert_architectures:
            return torch.tensor(0.0, device=device)
            
        # Encourage architectural diversity
        diversity_loss = 0.0
        architectures = list(expert_architectures.values())
        
        for i in range(len(architectures)):
            for j in range(i+1, len(architectures)):
                similarity = self._architecture_similarity(architectures[i], architectures[j])
                diversity_loss += similarity  # Penalize similarity
                
        # Encourage moderate sizes (not too big, not too small)
        size_regularization = 0.0
        target_size = 1000000  # 1M parameters target
        
        for depth, widths in architectures:
            total_params = sum(widths) * len(widths)  # Approximate parameter count
            size_penalty = (total_params - target_size) ** 2
            size_regularization += size_penalty
            
        return 0.01 * diversity_loss + 0.001 * size_regularization
    
    def _architecture_similarity(self, arch1: Tuple[int, List[int]], 
                                arch2: Tuple[int, List[int]]) -> float:
        """Compute similarity between two architectures"""
        depth1, widths1 = arch1
        depth2, widths2 = arch2
        
        # Depth similarity
        depth_sim = 1.0 - abs(depth1 - depth2) / self.max_layers
        
        # Width similarity (pad shorter architecture)
        max_depth = max(depth1, depth2)
        widths1_padded = widths1 + [0] * (max_depth - depth1)
        widths2_padded = widths2 + [0] * (max_depth - depth2)
        
        width_sim = 1.0 - sum(abs(w1 - w2) for w1, w2 in zip(widths1_padded, widths2_padded)) / (max_depth * self.max_neurons)
        
        return (depth_sim + width_sim) / 2.0

## Dynamic MoE Layer Implementation

This cell implements the complete Dynamic Topology MoE layer that integrates all components.

### Core Architecture:
- **Routing Network**: Linear layer that assigns tokens to experts
- **Dynamic Experts**: Each expert has variable depth and width
- **Memory Pool**: Efficient weight storage and retrieval
- **Controllers**: Manage architectural decisions and stability

### Forward Pass:
1. **Token Routing**: Assigns each token to an expert based on routing probabilities
2. **Expert Processing**: Each expert processes its assigned tokens through its current architecture
3. **Utilization Tracking**: Monitors activation patterns for architectural decisions
4. **Architecture Updates**: Periodically modifies expert architectures based on utilization

### Key Methods:
- `_process_through_experts()`: Handles token distribution and expert processing
- `_forward_through_expert()`: Processes tokens through a single expert's architecture
- `_maybe_update_architectures()`: Decides when to grow/shrink experts
- `get_architecture_stats()`: Provides statistics for monitoring and analysis

In [ ]:
class DynamicMoELayer(nn.Module):
    """
    Dynamic Topology Mixture of Experts layer.
    Each expert can adaptively change its architecture during training.
    """
    def __init__(self, hidden_size: int, num_experts: int = 8, max_layers: int = 6, 
                 max_neurons: int = 2048, dropout: float = 0.1):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_experts = num_experts
        self.max_layers = max_layers
        self.max_neurons = max_neurons
        self.dropout = dropout
        
        # Core components
        self.memory_pool = DynamicMemoryPool(num_experts, max_layers, max_neurons, hidden_size)
        self.gradient_flow = DynamicGradientFlow()
        self.arch_controller = SimpleArchitecturalController()
        self.load_balancer = CapacityAwareLoadBalancer(num_experts)
        self.stability = StabilityController()
        
        # Routing network
        self.router = nn.Linear(hidden_size, num_experts)
        
        # Layer normalization
        self.layer_norm = nn.LayerNorm(hidden_size)
        
        # Training step counter
        self.training_step = 0
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch_size, seq_len, hidden_dim = x.shape
        
        # 1. Route tokens to experts
        routing_logits = self.router(x)  # [batch_size, seq_len, num_experts]
        routing_probs = torch.softmax(routing_logits, dim=-1)
        
        # Get expert assignments
        expert_assignments = torch.argmax(routing_probs, dim=-1)  # [batch_size, seq_len]
        
        # 2. Process through experts
        expert_outputs = self._process_through_experts(x, expert_assignments)
        
        # 3. Update utilization statistics
        self._update_utilization_stats(expert_outputs, expert_assignments)
        
        # 4. Maybe update architectures
        if self.training and self.training_step % 1000 == 0:
            self._maybe_update_architectures()
            
        self.training_step += 1
        
        return expert_outputs
        
    def _process_through_experts(self, x: torch.Tensor, 
                                expert_assignments: torch.Tensor) -> torch.Tensor:
        """Process inputs through dynamic experts"""
        batch_size, seq_len, hidden_dim = x.shape
        outputs = torch.zeros_like(x)
        
        # Process each expert separately
        for expert_id in range(self.num_experts):
            expert_mask = (expert_assignments == expert_id)
            if not expert_mask.any():
                continue
                
            # Get inputs for this expert
            expert_inputs = x[expert_mask]  # [num_tokens, hidden_dim]
            
            # Process through expert
            expert_output = self._forward_through_expert(expert_id, expert_inputs)
            
            # Store outputs
            outputs[expert_mask] = expert_output
            
        return outputs
        
    def _forward_through_expert(self, expert_id: int, inputs: torch.Tensor) -> torch.Tensor:
        """Forward pass through a single expert"""
        current_input = inputs
        
        # Get current architecture
        depth, widths = self.memory_pool.get_expert_architecture(expert_id)
        
        if depth == 0:
            return inputs  # No active layers
            
        # Process through each layer
        for layer_id in range(depth):
            weights, bias = self.memory_pool.get_expert_weights(expert_id, layer_id)
            if weights is None:
                continue
                
            # Linear transformation
            layer_output = torch.matmul(current_input, weights.T) + bias
            
            # Activation function
            current_input = F.relu(layer_output)
            
            # Update utilization for this layer
            utilization = current_input.abs().mean().item()
            self.arch_controller.update_utilization(expert_id, layer_id, utilization)
            
        # Project back to original hidden size
        if current_input.shape[-1] != self.hidden_size:
            # Simple projection (could be made learnable)
            if current_input.shape[-1] > self.hidden_size:
                current_input = current_input[:, :self.hidden_size]
            else:
                padding = torch.zeros(current_input.shape[0], 
                                    self.hidden_size - current_input.shape[-1], 
                                    device=device)
                current_input = torch.cat([current_input, padding], dim=-1)
                
        return current_input
        
    def _update_utilization_stats(self, outputs: torch.Tensor, 
                                 expert_assignments: torch.Tensor):
        """Update utilization statistics for architectural decisions"""
        for expert_id in range(self.num_experts):
            expert_mask = (expert_assignments == expert_id)
            if expert_mask.any():
                expert_outputs = outputs[expert_mask]
                utilization = expert_outputs.abs().mean().item()
                depth, _ = self.memory_pool.get_expert_architecture(expert_id)
                if depth > 0:
                    self.arch_controller.update_utilization(expert_id, depth - 1, utilization)
                    
    def _maybe_update_architectures(self):
        """Periodically update expert architectures"""
        current_architectures = self.memory_pool.get_all_architectures()
        
        for expert_id in range(self.num_experts):
            depth, widths = current_architectures[expert_id]
            
            # Check if should grow
            if self.arch_controller.should_grow_expert(expert_id, depth, self.training_step):
                new_depth = depth + 1
                new_widths = widths + [512]  # Add layer with 512 neurons
                new_depth, new_widths = self.stability.enforce_architectural_limits(
                    expert_id, new_depth, new_widths
                )
                
                # Update architecture
                self.memory_pool.set_expert_architecture(expert_id, new_depth, new_widths)
                self.gradient_flow.handle_architecture_change(
                    expert_id, depth, new_depth, self.memory_pool
                )
                self.arch_controller.record_architecture_change(expert_id, self.training_step)
                self.load_balancer.invalidate_cache()
                
            # Check if should shrink
            elif self.arch_controller.should_shrink_expert(expert_id, depth, self.training_step):
                new_depth = max(1, depth - 1)
                new_widths = widths[:new_depth]
                new_depth, new_widths = self.stability.enforce_architectural_limits(
                    expert_id, new_depth, new_widths
                )
                
                # Update architecture
                self.memory_pool.set_expert_architecture(expert_id, new_depth, new_widths)
                self.gradient_flow.handle_architecture_change(
                    expert_id, depth, new_depth, self.memory_pool
                )
                self.arch_controller.record_architecture_change(expert_id, self.training_step)
                self.load_balancer.invalidate_cache()
                
    def get_load_balance_loss(self, routing_probs: torch.Tensor, 
                             expert_assignments: torch.Tensor) -> torch.Tensor:
        """Compute load balancing loss"""
        architectures = self.memory_pool.get_all_architectures()
        return self.load_balancer.compute_load_balance_loss(
            routing_probs, expert_assignments, architectures
        )
        
    def get_stability_loss(self) -> torch.Tensor:
        """Compute stability regularization loss"""
        architectures = self.memory_pool.get_all_architectures()
        return self.stability.compute_stability_loss(architectures)
        
    def get_architecture_stats(self) -> Dict:
        """Get statistics about current expert architectures"""
        architectures = self.memory_pool.get_all_architectures()
        stats = {
            'total_experts': self.num_experts,
            'expert_architectures': architectures,
            'average_depth': np.mean([depth for depth, _ in architectures.values()]),
            'average_width': np.mean([np.mean(widths) for _, widths in architectures.values()]),
            'total_parameters': sum(sum(widths) * len(widths) for _, widths in architectures.values())
        }
        return stats

## Enhanced GPT Model with Dynamic MoE

This cell integrates the Dynamic MoE layer into a complete GPT-style transformer model.

### Model Architecture:
- **Standard GPT Components**: Embeddings, positional encoding, attention, layer norm
- **Dynamic MoE Blocks**: Replace standard MLP with Dynamic MoE layers
- **Weight Tying**: Shares embedding and output projection weights

### Key Components:
- **LayerNorm**: Standard layer normalization
- **CausalSelfAttention**: Multi-head attention with causal masking
- **DynamicMoEBlock**: Transformer block with Dynamic MoE instead of MLP
- **DynamicMoEGPT**: Complete model with multiple MoE layers

### Training Features:
- **MoE Regularization**: Adds stability loss to prevent architectural collapse
- **Architecture Monitoring**: Tracks expert architectures across all layers
- **Generation Support**: Full text generation capabilities

### Model Configuration:
- Configurable depth, width, and expert count
- Supports mixed precision training
- Includes gradient clipping and weight decay

In [ ]:
class LayerNorm(nn.Module):
    def __init__(self, ndim, bias):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(ndim))
        self.bias = nn.Parameter(torch.zeros(ndim)) if bias else None
    def forward(self, x):
        return F.layer_norm(x, self.weight.shape, self.weight, self.bias, 1e-5)

class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.flash = hasattr(F, 'scaled_dot_product_attention')
        if not self.flash:
            self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size))
                                       .view(1, 1, config.block_size, config.block_size))

    def forward(self, x):
        B, T, C = x.size()
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)

        if self.flash:
            y = F.scaled_dot_product_attention(q, k, v, attn_mask=None, dropout_p=self.attn_dropout.p if self.training else 0.0, is_causal=True)
        else:
            att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
            att = att.masked_fill(self.bias[:, :, :T, :T] == 0, float('-inf'))
            att = F.softmax(att, dim=-1)
            att = self.attn_dropout(att)
            y = att @ v

        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.resid_dropout(self.c_proj(y))
        return y

class DynamicMoEBlock(nn.Module):
    """Block with Dynamic MoE instead of standard MLP"""
    def __init__(self, config):
        super().__init__()
        self.ln1 = LayerNorm(config.n_embd, config.bias)
        self.attn = CausalSelfAttention(config)
        self.ln2 = LayerNorm(config.n_embd, config.bias)
        self.moe = DynamicMoELayer(
            hidden_size=config.n_embd,
            num_experts=8,
            max_layers=6,
            max_neurons=2048,
            dropout=config.dropout
        )
        
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.moe(self.ln2(x))
        return x

@dataclass
class GPTConfig:
    block_size: int
    vocab_size: int
    n_layer: int
    n_head: int
    n_embd: int
    dropout: float = 0.0
    bias: bool = True

class DynamicMoEGPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.transformer = nn.ModuleDict(dict(
            wte=nn.Embedding(config.vocab_size, config.n_embd),
            wpe=nn.Embedding(config.block_size, config.n_embd),
            drop=nn.Dropout(config.dropout),
            h=nn.ModuleList([DynamicMoEBlock(config) for _ in range(config.n_layer)]),
            ln_f=LayerNorm(config.n_embd, config.bias),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight  # weight tying

        self.apply(self._init_weights)
        for pn, p in self.named_parameters():
            if pn.endswith('c_proj.weight'):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * config.n_layer))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        device = idx.device
        b, t = idx.size()
        assert t <= self.config.block_size
        pos = torch.arange(0, t, dtype=torch.long, device=device)

        tok_emb = self.transformer.wte(idx)
        pos_emb = self.transformer.wpe(pos)
        x = self.transformer.drop(tok_emb + pos_emb)
        
        # Track MoE losses
        moe_losses = []
        
        for block in self.transformer.h:
            x = block(x)
            # Collect MoE losses from each block
            if hasattr(block.moe, 'get_load_balance_loss') and hasattr(block.moe, 'get_stability_loss'):
                # This would need to be implemented to track routing during forward pass
                pass
                
        x = self.transformer.ln_f(x)

        if targets is not None:
            logits = self.lm_head(x)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1)
            
            # Add MoE regularization losses
            moe_reg_loss = 0.0
            for block in self.transformer.h:
                if hasattr(block.moe, 'get_stability_loss'):
                    moe_reg_loss += block.moe.get_stability_loss()
                    
            total_loss = loss + 0.01 * moe_reg_loss  # Small regularization weight
            return logits, total_loss
        else:
            logits = self.lm_head(x[:, [-1], :])
            return logits, None

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        """Generate tokens given a conditioning sequence."""
        for _ in range(max_new_tokens):
            idx_cond = idx if idx.size(1) <= self.config.block_size else idx[:, -self.config.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx
        
    def get_architecture_stats(self):
        """Get statistics about all MoE layers"""
        stats = {}
        for i, block in enumerate(self.transformer.h):
            if hasattr(block.moe, 'get_architecture_stats'):
                stats[f'layer_{i}'] = block.moe.get_architecture_stats()
        return stats

## Dataset Preparation

This cell prepares the TinyStories dataset for training the Dynamic MoE model.

### Dataset Overview:
- **TinyStories**: Synthetic dataset of simple stories for 3-4 year olds
- **Tokenization**: Uses GPT-2 tokenizer for consistent vocabulary
- **Binary Storage**: Saves tokenized data to disk for efficient loading

### Processing Pipeline:
1. **Dataset Loading**: Downloads TinyStories from HuggingFace
2. **Tokenization**: Converts text to token IDs using GPT-2 tokenizer
3. **Binary Conversion**: Saves tokens to memory-mapped binary files
4. **Efficient Storage**: Uses uint16 format for memory efficiency

### Key Features:
- **Parallel Processing**: Uses 8 processes for faster tokenization
- **Memory Mapping**: Enables efficient random access during training
- **Split Management**: Handles train/validation splits automatically

In [ ]:
# Install required packages
!pip install datasets tiktoken

from datasets import load_dataset
import tiktoken
import os
import numpy as np

# Load TinyStories dataset
ds = load_dataset("roneneldan/TinyStories")

# Tokenization setup
enc = tiktoken.get_encoding("gpt2")

def process(example):
    ids = enc.encode_ordinary(example['text'])
    out = {'ids': ids, 'len': len(ids)}
    return out

# Tokenize dataset
if not os.path.exists("train.bin"):
    tokenized = ds.map(
        process,
        remove_columns=['text'],
        desc="tokenizing the splits",
        num_proc=8,
    )
    
    # Save to binary files
    for split, dset in tokenized.items():
        arr_len = np.sum(dset['len'], dtype=np.uint64)
        filename = f'{split}.bin'
        dtype = np.uint16
        arr = np.memmap(filename, dtype=dtype, mode='w+', shape=(arr_len,))
        total_batches = 1024

        idx = 0
        for batch_idx in tqdm(range(total_batches), desc=f'writing {filename}'):
            batch = dset.shard(num_shards=total_batches, index=batch_idx, contiguous=True).with_format('numpy')
            arr_batch = np.concatenate(batch['ids'])
            arr[idx : idx + len(arr_batch)] = arr_batch
            idx += len(arr_batch)
        arr.flush()

print("Dataset preparation complete!")

## Training Configuration and Setup

This cell configures the training parameters and creates the model instance.

### Training Hyperparameters:
- **Learning Rate**: 1e-4 with warmup and cosine decay
- **Batch Size**: 16 with gradient accumulation (effective batch size = 256)
- **Context Length**: 128 tokens for sequence processing
- **Training Steps**: 10,000 iterations for demonstration

### Model Configuration:
- **Layers**: 4 transformer layers (reduced for faster training)
- **Heads**: 6 attention heads
- **Embedding**: 384 dimensions
- **Experts**: 8 experts per MoE layer

### Data Loading:
- **get_batch()**: Efficient batch generation with memory mapping
- **estimate_loss()**: Evaluation function for monitoring training progress
- **Device Management**: Automatic GPU/CPU handling with pin memory

### Key Optimizations:
- **Gradient Accumulation**: Simulates larger batch sizes
- **Mixed Precision**: Uses bfloat16/float16 for faster training
- **Memory Efficiency**: Pin memory for faster GPU transfer

In [ ]:
# Training configuration
learning_rate = 1e-4
max_iters = 10000  # Reduced for demonstration
warmup_steps = 500
min_lr = 5e-5
eval_iters = 200
batch_size = 16
block_size = 128
gradient_accumulation_steps = 16

# Model configuration
config = GPTConfig(
    vocab_size=50257,
    block_size=block_size,
    n_layer=4,  # Reduced for faster training
    n_head=6,
    n_embd=384,
    dropout=0.1,
    bias=True
)

# Create model
model = DynamicMoEGPT(config)
model = model.to(device)

print(f"Model created with {sum(p.numel() for p in model.parameters()):,} parameters")
print(f"Model configuration: {config}")

# Data loading function
def get_batch(split):
    if split == 'train':
        data = np.memmap('train.bin', dtype=np.uint16, mode='r')
    else:
        data = np.memmap('validation.bin', dtype=np.uint16, mode='r')
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([torch.from_numpy((data[i:i+block_size]).astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy((data[i+1:i+1+block_size]).astype(np.int64)) for i in ix])
    if device_type == 'cuda':
        x, y = x.pin_memory().to(device, non_blocking=True), y.pin_memory().to(device, non_blocking=True)
    else:
        x, y = x.to(device), y.to(device)
    return x, y

# Loss estimation function
def estimate_loss(model):
    out = {}
    model.eval()
    with torch.inference_mode():
        for split in ['train', 'val']:
            losses = torch.zeros(eval_iters)
            for k in range(eval_iters):
                X, Y = get_batch(split)
                with ctx:
                    logits, loss = model(X, Y)
                losses[k] = loss.item()
            out[split] = losses.mean()
    model.train()
    return out

## Training Loop Implementation

This cell implements the complete training loop with Dynamic MoE architecture monitoring.

### Training Components:
- **Optimizer**: AdamW with weight decay and gradient clipping
- **Scheduler**: Linear warmup followed by cosine decay
- **Mixed Precision**: Automatic gradient scaling for numerical stability

### Monitoring Features:
- **Loss Tracking**: Records training and validation losses
- **Architecture Evolution**: Tracks expert depth, width, and parameter counts
- **Model Checkpointing**: Saves best model based on validation loss

### Key Metrics:
- **Average Expert Depth**: How deep experts become over time
- **Average Expert Width**: How wide experts become over time
- **Total MoE Parameters**: Dynamic parameter count evolution
- **Learning Rate**: Current learning rate for monitoring

### Training Stability:
- **Gradient Clipping**: Prevents gradient explosion
- **Weight Decay**: Regularization to prevent overfitting
- **Architecture Cooldown**: Prevents rapid architectural oscillations

In [ ]:
# Optimizer and scheduler setup
from torch.optim.lr_scheduler import LinearLR, SequentialLR, CosineAnnealingLR

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, betas=(0.9, 0.95), weight_decay=0.1, eps=1e-9)
scheduler_warmup = LinearLR(optimizer, total_iters=warmup_steps)
scheduler_decay = CosineAnnealingLR(optimizer, T_max=max_iters - warmup_steps, eta_min=min_lr)
scheduler = SequentialLR(optimizer, schedulers=[scheduler_warmup, scheduler_decay], milestones=[warmup_steps])

scaler = torch.cuda.amp.GradScaler(enabled=(dtype == 'float16'))

# Training loop
best_val_loss = float('inf')
train_losses = []
val_losses = []
architecture_stats_history = []

print("Starting training...")

for iter in tqdm(range(max_iters)):
    # Evaluation
    if iter % eval_iters == 0 and iter != 0:
        losses = estimate_loss(model)
        train_losses.append(losses['train'])
        val_losses.append(losses['val'])
        
        # Get architecture statistics
        arch_stats = model.get_architecture_stats()
        architecture_stats_history.append(arch_stats)
        
        print(f"Iteration {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")
        print(f"Learning rate: {optimizer.param_groups[0]['lr']:.6f}")
        print(f"Average expert depth: {arch_stats.get('average_depth', 0):.2f}")
        print(f"Average expert width: {arch_stats.get('average_width', 0):.2f}")
        print(f"Total MoE parameters: {arch_stats.get('total_parameters', 0):,}")
        print("-" * 50)
        
        if losses['val'] < best_val_loss:
            best_val_loss = losses['val']
            torch.save(model.state_dict(), 'best_dynamic_moe_model.pt')

    # Training step
    X, y = get_batch('train')
    
    with ctx:
        logits, loss = model(X, y)
        loss = loss / gradient_accumulation_steps
        scaler.scale(loss).backward()

    if ((iter + 1) % gradient_accumulation_steps == 0) or (iter + 1 == max_iters):
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)
        scheduler.step()

print("Training complete!")

## Training Visualization and Analysis

This cell creates comprehensive visualizations to analyze the Dynamic MoE training process.

### Visualization Components:
1. **Loss Curves**: Training and validation loss over time
2. **Architecture Evolution**: How expert depths and widths change
3. **Parameter Evolution**: Dynamic parameter count tracking
4. **Expert Distribution**: Final architecture distribution across experts

### Key Insights:
- **Convergence Patterns**: How quickly the model learns
- **Architectural Adaptation**: How experts evolve based on data patterns
- **Parameter Efficiency**: Balance between model size and performance
- **Stability Analysis**: Whether architectural changes are stable

### Analysis Features:
- **Multi-panel Plot**: Comprehensive view of training dynamics
- **Grid Layout**: Organized visualization for easy interpretation
- **Color Coding**: Consistent color scheme for different metrics
- **Grid Lines**: Enhanced readability of plots

This visualization helps understand how the Dynamic MoE architecture adapts during training and whether the adaptive mechanisms are working effectively.

In [ ]:
# Plot training curves
plt.figure(figsize=(15, 10))

# Loss curves
plt.subplot(2, 3, 1)
plt.plot(train_losses, label='Train Loss', color='blue')
plt.plot(val_losses, label='Validation Loss', color='red')
plt.xlabel('Evaluation Steps')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.grid(True)

# Architecture evolution - Depth
plt.subplot(2, 3, 2)
avg_depths = [stats.get('average_depth', 0) for stats in architecture_stats_history]
plt.plot(avg_depths, label='Average Expert Depth', color='green')
plt.xlabel('Evaluation Steps')
plt.ylabel('Average Depth')
plt.title('Expert Architecture Evolution - Depth')
plt.legend()
plt.grid(True)

# Architecture evolution - Width
plt.subplot(2, 3, 3)
avg_widths = [stats.get('average_width', 0) for stats in architecture_stats_history]
plt.plot(avg_widths, label='Average Expert Width', color='orange')
plt.xlabel('Evaluation Steps')
plt.ylabel('Average Width')
plt.title('Expert Architecture Evolution - Width')
plt.legend()
plt.grid(True)

# Parameter count evolution
plt.subplot(2, 3, 4)
total_params = [stats.get('total_parameters', 0) for stats in architecture_stats_history]
plt.plot(total_params, label='Total MoE Parameters', color='purple')
plt.xlabel('Evaluation Steps')
plt.ylabel('Parameter Count')
plt.title('MoE Parameter Evolution')
plt.legend()
plt.grid(True)

# Expert architecture distribution (final state)
plt.subplot(2, 3, 5)
if architecture_stats_history:
    final_stats = architecture_stats_history[-1]
    expert_architectures = final_stats.get('expert_architectures', {})
    
    if expert_architectures:
        depths = [arch[0] for arch in expert_architectures.values()]
        avg_widths_per_expert = [np.mean(arch[1]) if arch[1] else 0 for arch in expert_architectures.values()]
        
        # Create scatter plot of depth vs average width
        plt.scatter(depths, avg_widths_per_expert, c=range(len(depths)), cmap='viridis', s=100, alpha=0.7)
        plt.xlabel('Expert Depth')
        plt.ylabel('Average Expert Width')
        plt.title('Final Expert Architecture Distribution')
        plt.colorbar(label='Expert ID')
        plt.grid(True)
    else:
        plt.text(0.5, 0.5, 'No architecture data available', ha='center', va='center', transform=plt.gca().transAxes)
        plt.title('Final Expert Architecture Distribution')
else:
    plt.text(0.5, 0.5, 'No training history available', ha='center', va='center', transform=plt.gca().transAxes)
    plt.title('Final Expert Architecture Distribution')

# Training efficiency metrics
plt.subplot(2, 3, 6)
if len(train_losses) > 1 and len(val_losses) > 1:
    # Calculate training efficiency (loss reduction per step)
    train_efficiency = [abs(train_losses[i] - train_losses[i-1]) for i in range(1, len(train_losses))]
    val_efficiency = [abs(val_losses[i] - val_losses[i-1]) for i in range(1, len(val_losses))]
    
    plt.plot(train_efficiency, label='Train Efficiency', color='blue', alpha=0.7)
    plt.plot(val_efficiency, label='Val Efficiency', color='red', alpha=0.7)
    plt.xlabel('Evaluation Steps')
    plt.ylabel('Loss Change per Step')
    plt.title('Training Efficiency Over Time')
    plt.legend()
    plt.grid(True)
else:
    plt.text(0.5, 0.5, 'Insufficient data for efficiency analysis', ha='center', va='center', transform=plt.gca().transAxes)
    plt.title('Training Efficiency Over Time')

plt.tight_layout()
plt.show()

# Print summary statistics
print("=== Training Summary ===")
if train_losses:
    print(f"Final Training Loss: {train_losses[-1]:.4f}")
    print(f"Final Validation Loss: {val_losses[-1]:.4f}")
    print(f"Best Validation Loss: {min(val_losses):.4f}")
    print(f"Training Steps: {len(train_losses)}")

if architecture_stats_history:
    final_stats = architecture_stats_history[-1]
    print(f"\n=== Final Architecture Statistics ===")
    print(f"Average Expert Depth: {final_stats.get('average_depth', 0):.2f}")
    print(f"Average Expert Width: {final_stats.get('average_width', 0):.2f}")
    print(f"Total MoE Parameters: {final_stats.get('total_parameters', 0):,}")
    
    expert_architectures = final_stats.get('expert_architectures', {})
    if expert_architectures:
        print(f"\n=== Individual Expert Architectures ===")
        for expert_id, (depth, widths) in expert_architectures.items():
            print(f"Expert {expert_id}: {depth} layers, widths={widths}")

# Additional detailed analysis
if len(architecture_stats_history) > 1:
    print(f"\n=== Architecture Evolution Analysis ===")
    initial_stats = architecture_stats_history[0]
    final_stats = architecture_stats_history[-1]
    
    depth_change = final_stats.get('average_depth', 0) - initial_stats.get('average_depth', 0)
    width_change = final_stats.get('average_width', 0) - initial_stats.get('average_width', 0)
    param_change = final_stats.get('total_parameters', 0) - initial_stats.get('total_parameters', 0)
    
    print(f"Depth Change: {depth_change:+.2f}")
    print(f"Width Change: {width_change:+.2f}")
    print(f"Parameter Change: {param_change:+,}")
    
    if param_change > 0:
        print("✓ Model grew during training")
    elif param_change < 0:
        print("✓ Model shrank during training")
    else:
        print("✓ Model size remained stable")